In [ ]:
# 1. Mount your Google Drive so Colab can see your files
from google.colab import drive
drive.mount('/content/drive')

# 2. Unzip the dataset into a new 'dataset' folder inside the shared directory
!unzip "/content/drive/MyDrive/amazon ml/6ab10eb3b23ba_student_resource.zip" -d "/content/drive/MyDrive/amazon ml/dataset/"


ValueError: mount failed

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install rapidfuzz unidecode

In [ ]:
!pip install rapidfuzz


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# New section

In [ ]:
import pandas as pd

# 1. Define the exact file paths from your unzip output
train_source1_path = "/content/drive/MyDrive/amazon ml/dataset/student_resource/dataset/train/train_source1.tsv"
ground_truth_path = "/content/drive/MyDrive/amazon ml/dataset/student_resource/dataset/train/train_ground_truth.tsv"

# 2. Load the TSV files. We must use sep='\t' to tell Pandas the data is separated by tabs, not commas.
df_train1 = pd.read_csv(train_source1_path, sep='\t')
df_labels = pd.read_csv(ground_truth_path, sep='\t')

# 3. Display the first 5 rows of each file to understand the columns
print("--- Train Source 1 Data ---")
display(df_train1.head())

print("\n--- Ground Truth (Target Prices) ---")
display(df_labels.head())

In [ ]:
# Load Source 2 and Source 3
train_source2_path = "/content/drive/MyDrive/amazon ml/dataset/student_resource/dataset/train/train_source2.tsv"
train_source3_path = "/content/drive/MyDrive/amazon ml/dataset/student_resource/dataset/train/train_source3.tsv"

df_train2 = pd.read_csv(train_source2_path, sep='\t')
df_train3 = pd.read_csv(train_source3_path, sep='\t')

# Let's check the size of each dataset to see what we are dealing with
print(f"Source 1 has {len(df_train1)} rows.")
print(f"Source 2 has {len(df_train2)} rows.")
print(f"Source 3 has {len(df_train3)} rows.")

# Display a sample from Source 2
print("\n--- Train Source 2 Data ---")
display(df_train2.head())

In [ ]:
import re
import pandas as pd

# 1. Take a small sample so Colab doesn't crash while we experiment
sample_s1 = df_train1.head(10000).copy()
sample_s2 = df_train2.head(10000).copy()

# 2. Create a function to clean the text
def clean_text(text):
    # Handle missing/NaN values
    if pd.isna(text):
        return ""

    # Convert everything to lowercase
    text = str(text).lower()

    # Remove punctuation but KEEP alphanumeric characters (including Hindi) and spaces
    text = re.sub(r'[^\w\s]', ' ', text)

    # Remove extra spaces that might have been created
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# 3. Apply the cleaning function to our sample data
sample_s2['clean_name'] = sample_s2['business_name'].apply(clean_text)
sample_s2['clean_address'] = sample_s2['business_address'].apply(clean_text)

print("--- Cleaned Source 2 Sample ---")
display(sample_s2[['business_name', 'clean_name', 'business_address', 'clean_address']].head())

In [ ]:
# 1. Install the unidecode library (run this cell once)
!pip install unidecode


In [ ]:
from unidecode import unidecode
import pandas as pd
import re

# 1. Upgraded cleaning function
def clean_and_romanize(text):
    if pd.isna(text):
        return ""

    # Ensure it's a string
    text = str(text)

    # Convert Hindi (and other scripts) to English characters FIRST
    text = unidecode(text)

    # Convert to lowercase
    text = text.lower()

    # Remove punctuation
    text = re.sub(r'[^\w\s]', ' ', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# 2. Apply the new function to the business names
sample_s2['romanized_name'] = sample_s2['business_name'].apply(clean_and_romanize)

print("--- Fixed Source 2 Sample ---")
display(sample_s2[['business_name', 'clean_name', 'romanized_name']].head())

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 1. Apply our working cleaning function to Source 1 as well!
print("Cleaning Source 1 sample... this might take a few seconds.")
sample_s1['romanized_name'] = sample_s1['business_name'].apply(clean_and_romanize)

# 2. Combine text to "teach" our vectorizer all the possible text variations
all_names = pd.concat([sample_s1['romanized_name'], sample_s2['romanized_name']]).fillna('')

# 3. Initialize TF-IDF.
# Pro-tip: We use analyzer='char_wb' (character level) instead of words.
# This helps catch fuzzy spelling mistakes like "raam" vs "ram".
vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4))
print("Building the vocabulary (TF-IDF)...")
vectorizer.fit(all_names)

# 4. Transform both datasets into mathematical matrices
s1_vectors = vectorizer.transform(sample_s1['romanized_name'].fillna(''))
s2_vectors = vectorizer.transform(sample_s2['romanized_name'].fillna(''))

# 5. Calculate matches! Let's test the first 3 businesses in Source 1
print("\n--- MATCHING RESULTS ---")
for i in range(3):
    query_vector = s1_vectors[i]

    # Calculate how similar this one Source 1 business is to ALL 10,000 Source 2 businesses
    similarities = cosine_similarity(query_vector, s2_vectors).flatten()

    # Find the index of the highest score
    best_match_idx = np.argmax(similarities)
    best_score = similarities[best_match_idx]

    print(f"\nLooking for: {sample_s1.iloc[i]['business_name']}")
    print(f"Top Match Found: {sample_s2.iloc[best_match_idx]['business_name']}")
    print(f"Match Score: {best_score:.2f} (1.00 is perfect)")

In [ ]:
# 1. Clean the addresses using the same function we built for the names
print("Cleaning addresses...")
sample_s1['romanized_address'] = sample_s1['business_address'].apply(clean_and_romanize)
sample_s2['romanized_address'] = sample_s2['business_address'].apply(clean_and_romanize)

# 2. Create a "Super String" combining name and address
sample_s1['combined_features'] = sample_s1['romanized_name'] + " " + sample_s1['romanized_address']
sample_s2['combined_features'] = sample_s2['romanized_name'] + " " + sample_s2['romanized_address']

# 3. Retrain TF-IDF on the combined strings
all_combined = pd.concat([sample_s1['combined_features'], sample_s2['combined_features']]).fillna('')
vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4))
print("Building new TF-IDF vocabulary...")
vectorizer.fit(all_combined)

s1_combined_vectors = vectorizer.transform(sample_s1['combined_features'].fillna(''))
s2_combined_vectors = vectorizer.transform(sample_s2['combined_features'].fillna(''))

# 4. Set a threshold (e.g., only accept matches that are 80% similar or higher)
MATCH_THRESHOLD = 0.80

print("\n--- NEW MATCHING RESULTS (NAME + ADDRESS) ---")
for i in range(3):
    query_vector = s1_combined_vectors[i]
    similarities = cosine_similarity(query_vector, s2_combined_vectors).flatten()

    best_match_idx = np.argmax(similarities)
    best_score = similarities[best_match_idx]

    print(f"\nSource 1 Business: {sample_s1.iloc[i]['business_name']} | {sample_s1.iloc[i]['business_address']}")
    print(f"Top Candidate: {sample_s2.iloc[best_match_idx]['business_name']} | {sample_s2.iloc[best_match_idx]['business_address']}")
    print(f"Similarity Score: {best_score:.2f}")

    # The F0.5 logic: Only accept if the score beats our threshold
    if best_score >= MATCH_THRESHOLD:
        print("Verdict: MATCH ACCEPTED ✅")
    else:
        print("Verdict: NO MATCH (Singleton) ❌")

In [ ]:
# 1. Grab the very first true match from the ground truth file
sample_s1_id = df_labels.iloc[0]['source1_entity_id']

# The matched_entity_ids is a comma-separated string; let's grab the first S2 ID
# We use a quick list comprehension to find the first ID that starts with 'S2-'
matches = df_labels.iloc[0]['matched_entity_ids'].split(',')
sample_s2_id = next((id for id in matches if id.startswith('S2-')), None)

if not sample_s2_id:
    print("No Source 2 match found for the first row. Try index 1 or 2.")
else:
    print(f"Testing Known Match: {sample_s1_id} <---> {sample_s2_id}")

    # 2. Fetch their actual text from the main dataframes
    s1_row = df_train1[df_train1['entity_id'] == sample_s1_id].iloc[0]
    s2_row = df_train2[df_train2['entity_id'] == sample_s2_id].iloc[0]

    # 3. Apply your exact cleaning pipeline to them
    s1_clean = clean_and_romanize(s1_row['business_name']) + " " + clean_and_romanize(s1_row['business_address'])
    s2_clean = clean_and_romanize(s2_row['business_name']) + " " + clean_and_romanize(s2_row['business_address'])

    # 4. Vectorize and score this specific pair
    pair_vectors = vectorizer.transform([s1_clean, s2_clean])
    score = cosine_similarity(pair_vectors[0], pair_vectors[1])[0][0]

    print(f"\nSource 1: {s1_row['business_name']} | {s1_row['business_address']}")
    print(f"Source 2: {s2_row['business_name']} | {s2_row['business_address']}")
    print(f"True Match Score: {score:.2f}")

    if score >= MATCH_THRESHOLD:
        print("Verdict: MATCH ACCEPTED ✅ (Engine working perfectly!)")
    else:
        print("Verdict: MATCH REJECTED ❌ (We may need to lower the threshold slightly)")

In [ ]:
# 0. Set our new, slightly more forgiving threshold!
MATCH_THRESHOLD = 0.75

# 1. We need TWO vectorizers now (one for names, one for addresses)
name_vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4))
name_vectorizer.fit(pd.concat([sample_s1['romanized_name'], sample_s2['romanized_name']]).fillna(''))

addr_vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4))
addr_vectorizer.fit(pd.concat([sample_s1['romanized_address'], sample_s2['romanized_address']]).fillna(''))

# 2. Clean the strings separately for our test pair
s1_name_clean = clean_and_romanize(s1_row['business_name'])
s2_name_clean = clean_and_romanize(s2_row['business_name'])

s1_addr_clean = clean_and_romanize(s1_row['business_address'])
s2_addr_clean = clean_and_romanize(s2_row['business_address'])

# 3. Calculate Name Score
name_vecs = name_vectorizer.transform([s1_name_clean, s2_name_clean])
name_score = cosine_similarity(name_vecs[0], name_vecs[1])[0][0]

# 4. Calculate Address Score & Handle Missing Data
if pd.isna(s1_row['business_address']) or pd.isna(s2_row['business_address']):
    final_score = name_score
    print("\n[Alert] Address missing in one of the records! Relying 100% on Name Score.")
else:
    addr_vecs = addr_vectorizer.transform([s1_addr_clean, s2_addr_clean])
    addr_score = cosine_similarity(addr_vecs[0], addr_vecs[1])[0][0]

    # Combine them: Give the Name 70% of the voting power, and Address 30%
    final_score = (name_score * 0.70) + (addr_score * 0.30)
    print(f"\nAddress Similarity Score: {addr_score:.2f}")

print(f"Name Similarity Score: {name_score:.2f}")
print(f"--- TRUE FINAL SCORE: {final_score:.2f} ---")

if final_score >= MATCH_THRESHOLD:
    print("Verdict: MATCH ACCEPTED ✅ (Engine working perfectly!)")
else:
    print("Verdict: MATCH REJECTED ❌ (We may need to lower the threshold slightly)")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from unidecode import unidecode
import re

# --- 1. UPDATED CLEANING FUNCTION ---
def clean_and_romanize(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = unidecode(text).lower()
    text = re.sub(r'[^\w\s]', ' ', text)

    # NEW: Added the transliterated garbage that unidecode generates for Hindi!
    stop_words = [r'\binc\b', r'\bcorp\b', r'\bcorporation\b', r'\bpvt\b',
                  r'\bprivate\b', r'\bltd\b', r'\blimited\b', r'\bllc\b', r'\bllp\b',
                  r'\bpraaivett\b', r'\blimittedd\b', r'\belelpii\b']

    for word in stop_words:
        text = re.sub(word, ' ', text)

    text = re.sub(r'\s+', ' ', text).strip()
    return text

# --- 2. EDGE CASE TEST SUITE ---
# Pairs designed to probe specific failure modes in entity resolution
test_cases = [
    {
        "case": "1. Standard Typo + Abbreviation",
        "expected": "ACCEPT",
        "s1_name": "Apex Logistics Private Limited",
        "s1_addr": "Plot 42, Okhla Industrial Area Phase 3, New Delhi",
        "s2_name": "Apex Logistcs Pvt Ltd",
        "s2_addr": "42 Okhla Ind Area Ph 3, Delhi"
    },
    {
        "case": "2. Devanagari Transliteration",
        "expected": "ACCEPT",
        "s1_name": "Ram Marketing Private Limited",
        "s1_addr": "KH NO. 570/13, New Delhi, West Delhi, Delhi",
        "s2_name": "राम मार्केटिंग प्राइवेट लिमिटेड",
        "s2_addr": "570 13 New Delhi West Delhi"
    },
    {
        "case": "3. The Chain Trap (Same Name, Different Address)",
        "expected": "REJECT",
        "s1_name": "Starbucks Coffee",
        "s1_addr": "1500 Broadway, Times Square, New York, NY",
        "s2_name": "Starbucks Coffee",
        "s2_addr": "102 1st Ave, Seattle, WA"
    },
    {
        "case": "4. The Mall Trap (Same Address, Different Business)",
        "expected": "REJECT",
        "s1_name": "Nike Flagship Store",
        "s1_addr": "DLF Mall of India, Sector 18, Noida",
        "s2_name": "Subway Sandwiches",
        "s2_addr": "DLF Mall of India, Sector 18, Noida"
    },
    {
        "case": "5. Both Addresses Missing (Pure Name Match)",
        "expected": "ACCEPT",
        "s1_name": "Horizon Tech Solutions",
        "s1_addr": np.nan,
        "s2_name": "Horizon Tech Solution Corp",
        "s2_addr": np.nan
    },
    {
        "case": "6. French Characters & Diacritics (Test Set Trap)",
        "expected": "ACCEPT",
        "s1_name": "Café de l'Étoile",
        "s1_addr": "12 Rue de Rivoli, 75004 Paris",
        "s2_name": "Cafe de l Etoile",
        "s2_addr": "12 Rue Rivoli, Paris"
    },
    {
        "case": "7. Word Transposition / Inversion",
        "expected": "ACCEPT",
        "s1_name": "National General Hospital & Pharmacy",
        "s1_addr": "Main Road, Sector 5, Salt Lake, Kolkata",
        "s2_name": "Pharmacy and Hospital National General",
        "s2_addr": "Sec 5, Salt Lake, Kolkata"
    }
]

# --- 3. VECTORIZER SETUP ---
all_test_names = []
all_test_addrs = []
for t in test_cases:
    all_test_names.extend([clean_and_romanize(t["s1_name"]), clean_and_romanize(t["s2_name"])])
    all_test_addrs.extend([clean_and_romanize(t["s1_addr"]), clean_and_romanize(t["s2_addr"])])

# Fit on test vocabulary combined with existing vocabulary
name_vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4)).fit(all_test_names)
addr_vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4)).fit(all_test_addrs)

# --- THE FINAL CALIBRATED THRESHOLD ---
MATCH_THRESHOLD = 0.65

# --- 4. RUN DIAGNOSTICS ---
print(f"{'Test Case':<50} | {'Name':<5} | {'Addr':<5} | {'Final':<5} | {'Outcome':<8} | {'Status'}")
print("-" * 90)

for t in test_cases:
    # Clean
    s1_n, s2_n = clean_and_romanize(t["s1_name"]), clean_and_romanize(t["s2_name"])
    s1_a, s2_a = clean_and_romanize(t["s1_addr"]), clean_and_romanize(t["s2_addr"])

    # Name similarity
    nv = name_vec.transform([s1_n, s2_n])
    name_score = cosine_similarity(nv[0], nv[1])[0][0]

    # Address similarity & fallback handling
    if pd.isna(t["s1_addr"]) or pd.isna(t["s2_addr"]) or s1_a == "" or s2_a == "":
        addr_score = np.nan
        final_score = name_score
    else:
        av = addr_vec.transform([s1_a, s2_a])
        addr_score = cosine_similarity(av[0], av[1])[0][0]

        # THE PERFECT BALANCE: 50% Name, 50% Address
        final_score = (name_score * 0.50) + (addr_score * 0.50)

    verdict = "ACCEPT" if final_score >= MATCH_THRESHOLD else "REJECT"
    status = "PASS ✅" if verdict == t["expected"] else "FAIL ⚠️"

    addr_str = f"{addr_score:.2f}" if not np.isnan(addr_score) else "None"
    print(f"{t['case']:<50} | {name_score:.2f}  | {addr_str:<5} | {final_score:.2f}  | {verdict:<8} | {status}")

In [ ]:
import pandas as pd

def calculate_competition_f05(ground_truth_dict, predictions_dict):
    """
    Calculates the exact macro-averaged F0.5 score used by the Amazon ML Challenge.

    ground_truth_dict: { 'S1-001': ['S2-001', 'S3-005'], 'S1-002': [] }
    predictions_dict:  { 'S1-001': ['S2-001'], 'S1-002': ['S2-999'] }
    """
    f05_scores = []

    for s1_id, true_matches in ground_truth_dict.items():
        # Get predictions, default to empty list if not found
        pred_matches = predictions_dict.get(s1_id, [])

        # Convert to sets for easy math
        true_set = set(true_matches)
        pred_set = set(pred_matches)

        # Singleton logic (Free points!)
        if len(true_set) == 0:
            if len(pred_set) == 0:
                f05_scores.append(1.0) # Correctly predicted no match
            else:
                f05_scores.append(0.0) # False positive on a singleton
            continue

        # Standard Precision & Recall math
        true_positives = len(true_set.intersection(pred_set))

        # If they guessed nothing, or got 0 right
        if true_positives == 0:
            f05_scores.append(0.0)
            continue

        precision = true_positives / len(pred_set)
        recall = true_positives / len(true_set)

        # The F0.5 Formula
        f05 = (1.25 * precision * recall) / ((0.25 * precision) + recall)
        f05_scores.append(f05)

    # Macro-average: average score across all entities
    final_score = sum(f05_scores) / len(f05_scores)
    return final_score

# --- QUICK SANITY TEST ---
# Let's test it exactly how the Challenge README example described!
# Ground truth: S1-00001 matches [S2-00047, S3-00812]
# Prediction: S1-00001 matches [S2-00047, S2-00193, S3-00812] (One false positive)
dummy_truth = {'S1-00001': ['S2-00047', 'S3-00812']}
dummy_preds = {'S1-00001': ['S2-00047', 'S2-00193', 'S3-00812']}

score = calculate_competition_f05(dummy_truth, dummy_preds)
print(f"Test Score: {score:.3f} (Should be exactly 0.714 based on the README)")

In [ ]:
!pip install unidecode

In [ ]:
import pandas as pd
import numpy as np
from rapidfuzz import fuzz
from unidecode import unidecode
import re
import os
import time

# ==========================================
# 1. CONFIGURATION
# ==========================================
MATCH_THRESHOLD = 0.65
CHUNK_SIZE = 1000

CANDIDATES_PATH = "/content/drive/MyDrive/amazon ml/dataset/student_resource/output/candidate_pairs.tsv"
S1_PATH = "/content/drive/MyDrive/amazon ml/dataset/student_resource/dataset/test/test_source1.tsv"
S2_PATH = "/content/drive/MyDrive/amazon ml/dataset/student_resource/dataset/test/test_source2.tsv"
S3_PATH = "/content/drive/MyDrive/amazon ml/dataset/student_resource/dataset/test/test_source3.tsv"
OUTPUT_PATH = "/content/drive/MyDrive/amazon ml/output/matching_results.tsv"

# ==========================================
# 2. PRE-LOAD DATABASES
# ==========================================
print("Loading target databases S1, S2, and S3 into memory...")
df_s1 = pd.read_csv(S1_PATH, sep='\t').set_index('entity_id')
df_targets = pd.concat([
    pd.read_csv(S2_PATH, sep='\t').set_index('entity_id'),
    pd.read_csv(S3_PATH, sep='\t').set_index('entity_id')
])
print("Databases loaded successfully!")

# ==========================================
# 3. FAST CLEANING & SCORING ENGINE (RAPIDFUZZ)
# ==========================================
def clean_and_romanize(text):
    if pd.isna(text): return ""
    text = str(text).strip()
    if not text: return ""
    text = unidecode(text).lower()
    text = re.sub(r'[^\w\s]', ' ', text)
    stop_words = [r'\binc\b', r'\bcorp\b', r'\bcorporation\b', r'\bpvt\b',
                  r'\bprivate\b', r'\bltd\b', r'\blimited\b', r'\bllc\b', r'\bllp\b',
                  r'\bpraaivett\b', r'\blimittedd\b', r'\belelpii\b']
    for word in stop_words: text = re.sub(word, ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

def score_candidates(s1_row, candidates_df):
    if candidates_df.empty: return []

    s1_name_clean = clean_and_romanize(s1_row['business_name'])
    s1_addr_clean = clean_and_romanize(s1_row['business_address'])

    accepted_matches = []

    for cand_id, cand_row in candidates_df.iterrows():
        cand_name_clean = clean_and_romanize(cand_row['business_name'])
        cand_addr_clean = clean_and_romanize(cand_row['business_address'])

        # RapidFuzz returns 0-100, divide by 100 to match our 0.0-1.0 scale
        name_score = fuzz.token_set_ratio(s1_name_clean, cand_name_clean) / 100.0

        if not s1_addr_clean or not cand_addr_clean:
            final_score = name_score
        else:
            addr_score = fuzz.token_set_ratio(s1_addr_clean, cand_addr_clean) / 100.0
            final_score = (name_score * 0.50) + (addr_score * 0.50)

        if final_score >= MATCH_THRESHOLD:
            accepted_matches.append(str(cand_row['entity_id']))

    return accepted_matches

# ==========================================
# 4. CHUNKING LOOP
# ==========================================
print(f"\nStarting chunked processing (Batch size: {CHUNK_SIZE})...")
start_time = time.time()

# Ensure output directory exists and write the TSV headers
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
with open(OUTPUT_PATH, 'w') as f:
    f.write("source1_entity_id\tmatched_entity_ids\n")

chunk_counter = 0

for chunk in pd.read_csv(CANDIDATES_PATH, sep='\t', chunksize=CHUNK_SIZE):
    chunk_results = []

    for _, row in chunk.iterrows():
        s1_id = row['source1_entity_id']

        if pd.isna(row['candidate_entity_ids']) or str(row['candidate_entity_ids']).strip() == "":
            chunk_results.append({'source1_entity_id': s1_id, 'matched_entity_ids': ""})
            continue

        try:
            s1_row = df_s1.loc[s1_id]
        except KeyError:
            continue

        cand_ids = str(row['candidate_entity_ids']).split(',')
        valid_cand_ids = [cid for cid in cand_ids if cid in df_targets.index]

        if not valid_cand_ids:
            chunk_results.append({'source1_entity_id': s1_id, 'matched_entity_ids': ""})
            continue

        cands_data = df_targets.loc[valid_cand_ids].copy()
        cands_data['entity_id'] = cands_data.index

        final_matches = score_candidates(s1_row, cands_data)

        match_str = ",".join(final_matches)
        chunk_results.append({'source1_entity_id': s1_id, 'matched_entity_ids': match_str})

    chunk_df = pd.DataFrame(chunk_results)
    chunk_df.to_csv(OUTPUT_PATH, mode='a', header=False, index=False, sep='\t', quoting=3)

    chunk_counter += 1
    elapsed = time.time() - start_time
    print(f"Processed {(chunk_counter * CHUNK_SIZE)} S1 records... Elapsed time: {elapsed:.2f}s")

print(f"\nSUCCESS! Finished processing all S1 entities.")
print(f"Final file safely stored at: {OUTPUT_PATH}")

In [ ]:
import pandas as pd
import numpy as np
from rapidfuzz import process, fuzz
from unidecode import unidecode
import re
import os
import time

# ==========================================
# 1. BENCHMARK CONFIGURATION
# ==========================================
MATCH_THRESHOLD = 0.65
CHUNK_SIZE = 50000  # Testing a massive 50k batch per ChatGPT's recommendation

CANDIDATES_PATH = "/content/drive/MyDrive/amazon ml/dataset/student_resource/output/candidate_pairs.tsv"
S1_PATH = "/content/drive/MyDrive/amazon ml/dataset/student_resource/dataset/test/test_source1.tsv"
S2_PATH = "/content/drive/MyDrive/amazon ml/dataset/student_resource/dataset/test/test_source2.tsv"
S3_PATH = "/content/drive/MyDrive/amazon ml/dataset/student_resource/dataset/test/test_source3.tsv"
OUTPUT_PATH = "/content/drive/MyDrive/amazon ml/output/matching_results_benchmark.tsv"

# ==========================================
# 2. PRE-CLEANING DATABASES (RUNS ONCE)
# ==========================================
def clean_and_romanize_series(series):
    def clean_text(text):
        if pd.isna(text): return ""
        text = str(text).strip()
        if not text: return ""
        text = unidecode(text).lower()
        text = re.sub(r'[^\w\s]', ' ', text)
        stop_words = [r'\binc\b', r'\bcorp\b', r'\bcorporation\b', r'\bpvt\b',
                      r'\bprivate\b', r'\bltd\b', r'\blimited\b', r'\bllc\b', r'\bllp\b',
                      r'\bpraaivett\b', r'\blimittedd\b', r'\belelpii\b']
        for word in stop_words:
            text = re.sub(word, ' ', text)
        return re.sub(r'\s+', ' ', text).strip()
    return series.apply(clean_text)

print("Loading and Pre-Cleaning databases S1, S2, and S3 (This may take a minute)...")
df_s1 = pd.read_csv(S1_PATH, sep='\t').set_index('entity_id')
df_targets = pd.concat([
    pd.read_csv(S2_PATH, sep='\t').set_index('entity_id'),
    pd.read_csv(S3_PATH, sep='\t').set_index('entity_id')
])

df_s1['clean_name'] = clean_and_romanize_series(df_s1['business_name'])
df_s1['clean_addr'] = clean_and_romanize_series(df_s1['business_address'])
df_targets['clean_name'] = clean_and_romanize_series(df_targets['business_name'])
df_targets['clean_addr'] = clean_and_romanize_series(df_targets['business_address'])
print("Pre-cleaning complete! Starting RapidFuzz cpdist Engine...")

# ==========================================
# 3. VECTORIZED SCORING ENGINE
# ==========================================
start_time = time.time()
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
with open(OUTPUT_PATH, 'w') as f:
    f.write("source1_entity_id\tmatched_entity_ids\n")

chunk_counter = 0

for chunk in pd.read_csv(CANDIDATES_PATH, sep='\t', chunksize=CHUNK_SIZE):

    # Drop rows with no candidates at all
    chunk = chunk[chunk['candidate_entity_ids'].notna()].copy()

    # Explode candidate IDs so every row is a 1-to-1 pair
    chunk['cand_id_list'] = chunk['candidate_entity_ids'].astype(str).str.split(',')
    exploded = chunk.explode('cand_id_list')
    exploded['cand_id'] = exploded['cand_id_list'].str.strip()
    exploded = exploded[exploded['cand_id'] != ""]

    if exploded.empty:
        continue

    # Map the pre-cleaned names and addresses to the exploded pairs
    s1_names = exploded['source1_entity_id'].map(df_s1['clean_name']).fillna("").to_numpy(dtype=str)
    s1_addrs = exploded['source1_entity_id'].map(df_s1['clean_addr']).fillna("").to_numpy(dtype=str)

    cand_names = exploded['cand_id'].map(df_targets['clean_name']).fillna("").to_numpy(dtype=str)
    cand_addrs = exploded['cand_id'].map(df_targets['clean_addr']).fillna("").to_numpy(dtype=str)

    # ----------------------------------------------------
    # C++ PARALLEL SCORING (The Game Changer)
    # ----------------------------------------------------
    name_scores = process.cpdist(
        s1_names,
        cand_names,
        scorer=fuzz.token_set_ratio,
        workers=-1,
        dtype=np.uint8
    )

    addr_scores = process.cpdist(
        s1_addrs,
        cand_addrs,
        scorer=fuzz.token_set_ratio,
        workers=-1,
        dtype=np.uint8
    )

    # Vectorized 50/50 Math and Missing Address Fallback
    addr_empty_mask = (s1_addrs == "") | (cand_addrs == "")
    final_scores = np.where(
        addr_empty_mask,
        name_scores.astype(np.float32) / 100.0,
        (0.5 * name_scores.astype(np.float32) + 0.5 * addr_scores.astype(np.float32)) / 100.0
    )

    # Filter and aggregate
    exploded['score'] = final_scores
    matches = exploded[exploded['score'] >= MATCH_THRESHOLD]
    agg_matches = matches.groupby('source1_entity_id')['cand_id'].apply(lambda x: ','.join(x)).reset_index()
    agg_matches.rename(columns={'cand_id': 'matched_entity_ids'}, inplace=True)

    # Merge back to ensure S1 entities with zero matches output a blank string
    final_chunk_df = pd.merge(chunk[['source1_entity_id']], agg_matches, on='source1_entity_id', how='left')
    final_chunk_df['matched_entity_ids'] = final_chunk_df['matched_entity_ids'].fillna("")

    # Save Output
    final_chunk_df.to_csv(OUTPUT_PATH, mode='a', header=False, index=False, sep='\t', quoting=3)

    chunk_counter += 1
    elapsed = time.time() - start_time
    print(f"Processed {(chunk_counter * CHUNK_SIZE)} S1 records... Elapsed time: {elapsed:.2f}s")

    # STOP AFTER 100,000 RECORDS FOR THE BENCHMARK
    if chunk_counter >= 2:
        print("\nBENCHMARK COMPLETE! Provide this elapsed time.")
        break

In [ ]:
import pandas as pd
import numpy as np
from rapidfuzz import process, fuzz
from unidecode import unidecode
import re
import os
import time

# ==========================================
# 1. BENCHMARK CONFIGURATION
# ==========================================
MATCH_THRESHOLD = 0.65
CHUNK_SIZE = 10000  # Memory-safe size for Colab's 12GB RAM limit

CANDIDATES_PATH = "/content/drive/MyDrive/amazon ml/dataset/student_resource/output/candidate_pairs.tsv"
S1_PATH = "/content/drive/MyDrive/amazon ml/dataset/student_resource/dataset/test/test_source1.tsv"
S2_PATH = "/content/drive/MyDrive/amazon ml/dataset/student_resource/dataset/test/test_source2.tsv"
S3_PATH = "/content/drive/MyDrive/amazon ml/dataset/student_resource/dataset/test/test_source3.tsv"
OUTPUT_PATH = "/content/drive/MyDrive/amazon ml/output/matching_results_benchmark.tsv"

# ==========================================
# 2. PRE-CLEANING DATABASES (RUNS ONCE)
# ==========================================
def clean_and_romanize_series(series):
    def clean_text(text):
        if pd.isna(text): return ""
        text = str(text).strip()
        if not text: return ""
        text = unidecode(text).lower()
        text = re.sub(r'[^\w\s]', ' ', text)
        stop_words = [r'\binc\b', r'\bcorp\b', r'\bcorporation\b', r'\bpvt\b',
                      r'\bprivate\b', r'\bltd\b', r'\blimited\b', r'\bllc\b', r'\bllp\b',
                      r'\bpraaivett\b', r'\blimittedd\b', r'\belelpii\b']
        for word in stop_words:
            text = re.sub(word, ' ', text)
        return re.sub(r'\s+', ' ', text).strip()
    return series.apply(clean_text)

print("Loading and Pre-Cleaning databases S1, S2, and S3 (This may take ~10 minutes)...")
df_s1 = pd.read_csv(S1_PATH, sep='\t').set_index('entity_id')
df_targets = pd.concat([
    pd.read_csv(S2_PATH, sep='\t').set_index('entity_id'),
    pd.read_csv(S3_PATH, sep='\t').set_index('entity_id')
])

df_s1['clean_name'] = clean_and_romanize_series(df_s1['business_name'])
df_s1['clean_addr'] = clean_and_romanize_series(df_s1['business_address'])
df_targets['clean_name'] = clean_and_romanize_series(df_targets['business_name'])
df_targets['clean_addr'] = clean_and_romanize_series(df_targets['business_address'])
print("Pre-cleaning complete! Starting RapidFuzz cpdist Engine...")

# ==========================================
# 3. VECTORIZED SCORING ENGINE
# ==========================================
start_time = time.time()
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
with open(OUTPUT_PATH, 'w') as f:
    f.write("source1_entity_id\tmatched_entity_ids\n")

chunk_counter = 0

for chunk in pd.read_csv(CANDIDATES_PATH, sep='\t', chunksize=CHUNK_SIZE):

    # Drop rows with no candidates at all
    chunk = chunk[chunk['candidate_entity_ids'].notna()].copy()

    # Explode candidate IDs so every row is a 1-to-1 pair
    chunk['cand_id_list'] = chunk['candidate_entity_ids'].astype(str).str.split(',')
    exploded = chunk.explode('cand_id_list')
    exploded['cand_id'] = exploded['cand_id_list'].str.strip()
    exploded = exploded[exploded['cand_id'] != ""]

    if exploded.empty:
        continue

    # Map the pre-cleaned names and addresses to the exploded pairs
    s1_names = exploded['source1_entity_id'].map(df_s1['clean_name']).fillna("").to_numpy(dtype=str)
    s1_addrs = exploded['source1_entity_id'].map(df_s1['clean_addr']).fillna("").to_numpy(dtype=str)

    cand_names = exploded['cand_id'].map(df_targets['clean_name']).fillna("").to_numpy(dtype=str)
    cand_addrs = exploded['cand_id'].map(df_targets['clean_addr']).fillna("").to_numpy(dtype=str)

    # ----------------------------------------------------
    # C++ PARALLEL SCORING
    # ----------------------------------------------------
    name_scores = process.cpdist(
        s1_names,
        cand_names,
        scorer=fuzz.token_set_ratio,
        workers=-1,
        dtype=np.uint8
    )

    addr_scores = process.cpdist(
        s1_addrs,
        cand_addrs,
        scorer=fuzz.token_set_ratio,
        workers=-1,
        dtype=np.uint8
    )

    # Vectorized 50/50 Math and Missing Address Fallback
    addr_empty_mask = (s1_addrs == "") | (cand_addrs == "")
    final_scores = np.where(
        addr_empty_mask,
        name_scores.astype(np.float32) / 100.0,
        (0.5 * name_scores.astype(np.float32) + 0.5 * addr_scores.astype(np.float32)) / 100.0
    )

    # Filter and aggregate
    exploded['score'] = final_scores
    matches = exploded[exploded['score'] >= MATCH_THRESHOLD]
    agg_matches = matches.groupby('source1_entity_id')['cand_id'].apply(lambda x: ','.join(x)).reset_index()
    agg_matches.rename(columns={'cand_id': 'matched_entity_ids'}, inplace=True)

    # Merge back to ensure S1 entities with zero matches output a blank string
    final_chunk_df = pd.merge(chunk[['source1_entity_id']], agg_matches, on='source1_entity_id', how='left')
    final_chunk_df['matched_entity_ids'] = final_chunk_df['matched_entity_ids'].fillna("")

    # Save Output
    final_chunk_df.to_csv(OUTPUT_PATH, mode='a', header=False, index=False, sep='\t', quoting=3)

    chunk_counter += 1
    elapsed = time.time() - start_time
    print(f"Processed {(chunk_counter * CHUNK_SIZE)} S1 records... Elapsed time: {elapsed:.2f}s")

    # STOP AFTER 100,000 RECORDS (10 Chunks)
    if chunk_counter >= 10:
        print("\nBENCHMARK COMPLETE! Provide this elapsed time.")
        break